In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from datetime import datetime
import itertools

import numpy as np
from numpy.random import seed, binomial, weibull, exponential, normal
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from relife.lifetime_model import SemiParametricAcceleratedFailureTime

In [ ]:
root_data_path = Path(r"D:\Projets\RTE\ReLife\data")

# Fonctions

In [ ]:
def get_simulated_dataset(
        weibull_shape: float,
        truncation_exponential_scale: float | None,
        params: np.ndarray,
        N: int,
        nseed: int
):
    assert (params.ndim == 1) and (len(params) == 2), "params must be a vector of 2 parameters"
    seed(nseed)
    covar1 = binomial(n=1, p=0.5, size=N)
    covar2 = normal(scale=0.1, size=N)
    covar = np.concat((covar1[:, None], covar2[:, None]), axis=1)
    g = np.exp(params[0] * covar1 + params[1] * covar2)
    yy = g * weibull(a=weibull_shape, size=N)
    cc = exponential(size=N)
    time = np.minimum(yy, cc)
    event = yy <= cc
    if truncation_exponential_scale is None:
        tr = None
        entry = None
    else:
        assert truncation_exponential_scale < 1, "you must ensure truncation parameterization is consistent with right-censoring one"
        tr = exponential(scale=truncation_exponential_scale, size=N)
        entry = np.minimum(time, tr)
    return time, covar, event, entry, yy, cc, tr, params

In [ ]:
def plot_overlapping_hist_matplotlib(df, bins=30, alpha=0.5):
    """
    Plot overlapping histograms of all numeric columns using matplotlib.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe
    bins : int
        Number of histogram bins
    alpha : float
        Transparency level for overlap
    """
    numeric_df = df.select_dtypes(include="number")

    plt.figure(figsize=(10, 6))

    for col in numeric_df.columns:
        plt.hist(numeric_df[col].dropna(),
                 bins=bins,
                 alpha=alpha,
                 label=col)

    plt.xlabel("Value")
    plt.ylabel("Frequency")
    plt.title("Overlapping Histograms (Matplotlib)")
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
def expandgrid(*itrs) -> list[list]:
    """
    Generate a Cartesian product of the input iterables.

    This function takes any number of iterables and returns a list of lists
    containing the Cartesian product of the provided iterables. The order of the
    input iterables is reversed to maintain compatibility with R's expand.grid.

    Parameters:
        *itrs: Variable length argument list of iterables.

    Returns:
        A list of lists representing the Cartesian product.
    """
    product = list(itertools.product(*reversed(itrs)))
    return [[x[i] for x in product] for i in range(len(itrs))][::-1]

# Récupération des données

## Données d'isolateur

In [ ]:
# Données chaines d'isolateur
relife_csv_datapath = Path(r"D:\Projets\RTE\ReLife\relife\relife\data\csv") # TODO: changer le path en dur
time, event, entry, *args = np.loadtxt(relife_csv_datapath / "insulator_string.csv", delimiter=",", skiprows=1,
                                       unpack=True)
covar = np.column_stack(args)

## Données "Channing"

In [ ]:
# Données Channing
channing_data = pd.read_csv(root_data_path / "channing.csv", sep=";", decimal=",")
channing_data = (
    channing_data
    .drop(columns="time")
    .rename(columns={"exit": "time", "cens": "event"})
)
time, event, entry = channing_data["time"].values, channing_data["event"].astype(float).values, channing_data["entry"].values
covar = (channing_data[["sex"]] == "Male").astype(float).values

## Données simulées

In [ ]:
# Plot theoretical lifetime and right-censoring time sample distributions
simulation_case_study_kwargs = {
    "nseed": 4, "N": 5000, "weibull_shape": 1.5, "truncation_exponential_scale": 0.25, "params": np.array([1, 2.3])
}

time, covar, event, entry, lifetime, right_censoring, left_truncature, _ = get_simulated_dataset(**simulation_case_study_kwargs)

plot_overlapping_hist_matplotlib(
    pd.DataFrame({"time": time, "entry": entry, "time - entry": time - entry})
)

# Fit du modèle

In [ ]:
# Model
model = SemiParametricAcceleratedFailureTime()

## Sur 1 jeu de données

In [ ]:
# Test fit
N = len(covar)

model.fit(
     time=time[:N], covar=covar[:N], event=event[:N], entry=entry[:N] if entry is not None else None
)
print(model.params)

## Sur simulations

### Grille de paramètres

In [ ]:
# Définir une grille de paramètres de simulation
seeds = range(5)
n_range = [5000]
weibull_shape = [0.5, 1, 1.5, 5]
truncation_exponential_scale = [None, 0.25, 0.5]
params = [np.array([1, 2.3]), np.array([5, 1.3])]

simulation_grid_params = pd.DataFrame(
    expandgrid(seeds, n_range, weibull_shape, truncation_exponential_scale, params),
    index=["nseed", "N", "weibull_shape", "truncation_exponential_scale", "params"]
).T
simulation_grid_params

### Simulations et collecte des résultats de fit

In [ ]:
# Collecte des estimations de paramètres
file_path = root_data_path / "simu_res_2.csv"

simu_res = []
for idx, simu_params in simulation_grid_params.iterrows():
    simu_params_dict = simu_params.to_dict()
    print(f"simu {idx}: {simu_params_dict}")
    # From simulation params to simulation dataset
    time, covar, event, entry, _, _, _, params = get_simulated_dataset(**simu_params_dict)
    # Model fitting
    start_fit_time = datetime.now()
    model.fit(
        time=time, covar=covar, event=event, entry=entry if entry is not None else None
    )
    fit_duration_mn = (datetime.now() - start_fit_time).total_seconds() / 60
    # Collect results
    simu_params_dict.update({"params_est": model.params, "fit_duration_mn": fit_duration_mn})
    simu_res.append(pd.Series(simu_params_dict))

simu_res = pd.DataFrame(simu_res)
simu_res.to_csv(file_path, sep=";", decimal=",", index=False)

### Plot

In [ ]:
# Plot
filtering = simu_res["truncation_exponential_scale"] == 0.5

simu_res["truncation_exponential_scale"] = simu_res["truncation_exponential_scale"].replace(np.nan, 0)
params_df = pd.DataFrame(simu_res["params"].tolist(), index=simu_res.index)
distrib_params = (
    simu_res
    .loc[filtering, ["weibull_shape", "truncation_exponential_scale"]]
    .join(params_df[filtering])
    .drop_duplicates()
    .reset_index(drop=True)
)

fig, ax = plt.subplots(distrib_params.shape[0], 2, sharex=True, sharey=False, figsize=(8,12))
for ir, one_distrib in distrib_params.iterrows():
    one_distrib_simu_res = simu_res[
        (simu_res["weibull_shape"] == one_distrib["weibull_shape"])
        & (simu_res["truncation_exponential_scale"] == one_distrib["truncation_exponential_scale"])
        & (params_df[0] == one_distrib[0])
        & (params_df[1] == one_distrib[1])
    ]
    params_est = pd.DataFrame(
        one_distrib_simu_res["params_est"].tolist(), index=one_distrib_simu_res.index
    ).join(one_distrib_simu_res[["N", "nseed"]])
    for covar_i in range(params_df.shape[1]):
        sns.barplot(
            data=params_est[["N", "nseed", covar_i]].rename(columns={"nseed": "seed", covar_i: f"covar{covar_i}"}),
            x="N",
            y=f"covar{covar_i}",
            hue="seed",
            ax=ax[ir, covar_i],
            legend=False
        )
        ax[ir, covar_i].axhline(y=one_distrib[covar_i], color="black", linestyle="dashed")
        ax[ir, covar_i].set_title(f"weibull_shape: {one_distrib['weibull_shape']}, truncation_exponential_scale: {one_distrib['truncation_exponential_scale']}\ncovar{covar_i}: {one_distrib[covar_i]}", fontsize=8)
fig.tight_layout()